# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YassinKhodir/yassin-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook tests one deliberately simple model for the **Refresh / Content Opportunity Scoring** lane. It uses the same March page frame and the frozen ML-07 rule, then compares both on the same held-out clients.

## 1. Method choice and why

I use **logistic regression** because the outcome is binary and the output probability can rank pages for review. It is fast, readable, and gives a useful first test of whether learning weights improves on the fixed baseline. I am not adding a complex model just to make the notebook look advanced.

In [1]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

required = {
    "duckdb": "duckdb", "pandas": "pandas", "numpy": "numpy",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data_root = os.environ.get("FLYRANK_DATA_ROOT")
if data_root:
    REL = data_root.rstrip("/")
else:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = None
    if not hf_token:
        raise RuntimeError("Add a Hugging Face READ token as a Colab Secret named HF_TOKEN; never paste it into this public notebook.")
    REL = "hf://datasets/FlyRank/internship-warehouse"

con = duckdb.connect()
if not data_root:
    escaped_token = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
    del hf_token, escaped_token

MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
feature_query = f"""
WITH early AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS early_impressions,
        SUM(gsc_clicks) AS early_clicks,
        COUNT(*) AS early_observed_days,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS early_active_days,
        SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                 THEN gsc_avg_position * gsc_impressions ELSE 0 END)
          / NULLIF(SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                            THEN gsc_impressions ELSE 0 END), 0) AS early_avg_position,
        STDDEV_SAMP(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS early_position_volatility
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
late AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS late_impressions,
           COUNT(*) AS late_observed_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1, 2
)
SELECT
    e.client_hash_id, e.content_hash_id,
    LN(1 + e.early_impressions) AS log_early_impressions,
    1.0 * e.early_clicks / NULLIF(e.early_impressions, 0) AS early_ctr,
    e.early_active_days, e.early_avg_position, e.early_position_volatility,
    e.early_impressions,
    CAST((1.0 * l.late_impressions / NULLIF(l.late_observed_days, 0))
         < 0.8 * (1.0 * e.early_impressions / NULLIF(e.early_observed_days, 0)) AS INTEGER)
      AS is_future_decline
FROM early e
JOIN late l USING (client_hash_id, content_hash_id)
ORDER BY e.client_hash_id, e.content_hash_id
"""

frame = con.sql(feature_query).df()
features = [
    "log_early_impressions", "early_ctr", "early_active_days",
    "early_avg_position", "early_position_volatility",
]
print(f"Rows: {len(frame):,}; clients: {frame['client_hash_id'].nunique()}; sklearn: {sklearn.__version__}")

Rows: 77,400; clients: 38; sklearn: 1.8.0


## 2. Split design

I hold out entire clients with a fixed random seed. This is harder than mixing pages from every client into both sets because the model cannot learn a client's typical pattern and see that same client during testing. All five features stop on March 15; the observed label comes from March 16–31.

In [2]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(frame, frame["is_future_decline"], groups=frame["client_hash_id"]))
train = frame.iloc[train_idx].copy()
test = frame.iloc[test_idx].copy()

assert set(train["client_hash_id"]).isdisjoint(set(test["client_hash_id"]))
print(f"Train: {len(train):,} rows from {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} rows from {test['client_hash_id'].nunique()} clients")
print(f"Held-out-client base rate: {test['is_future_decline'].mean():.3f}")

Train: 61,624 rows from 28 clients
Test:  15,776 rows from 10 clients
Held-out-client base rate: 0.233


## 3. Train and compare with the frozen baseline

Both methods rank the same held-out rows and use the same later-window label. The baseline score is copied exactly from ML-07. The model uses median imputation, standardization, and logistic regression with seed 42. I report precision at several queue sizes because the decision is “which pages first?”, plus ROC-AUC as a broad ranking check.

In [3]:
test["volume_points"] = pd.cut(
    test["early_impressions"], [-1, 299, 999, 2999, 9999, np.inf], labels=[0, 1, 2, 3, 4]
).astype(int)
test["position_points"] = np.select(
    [
        test["early_avg_position"].between(20, 50, inclusive="right"),
        test["early_avg_position"].between(10, 20, inclusive="right"),
        test["early_avg_position"].between(3, 10, inclusive="right"),
    ], [3, 2, 1], default=0,
)
test["baseline_score"] = test["volume_points"] + test["position_points"]

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)
model.fit(train[features], train["is_future_decline"])
test["model_score"] = model.predict_proba(test[features])[:, 1]

def precision_at_k(data, score, k):
    ranked = data.sort_values(
        [score, "early_impressions", "early_avg_position", "content_hash_id"],
        ascending=[False, False, True, True],
    )
    return ranked.head(k)["is_future_decline"].mean()

rows = [{"method": "base rate", "precision@10": test["is_future_decline"].mean(),
         "precision@50": test["is_future_decline"].mean(), "precision@100": test["is_future_decline"].mean(),
         "roc_auc": 0.5}]
for name, score in [("frozen rule", "baseline_score"), ("logistic regression", "model_score")]:
    rows.append({
        "method": name,
        "precision@10": precision_at_k(test, score, 10),
        "precision@50": precision_at_k(test, score, 50),
        "precision@100": precision_at_k(test, score, 100),
        "roc_auc": roc_auc_score(test["is_future_decline"], test[score]),
    })
comparison = pd.DataFrame(rows).round(3)
comparison

                method  precision@10  precision@50  precision@100  roc_auc
0            base rate         0.233         0.233          0.233     0.50
1          frozen rule         0.300         0.260          0.150     0.43
2  logistic regression         0.600         0.340          0.310     0.51

## 4. Errors and interpretation

The model improves precision at the top of this held-out queue, but its ROC-AUC is only slightly above 0.50. The honest reading is narrow: the learned score helps prioritize this particular top slice, while overall separation remains weak. It is not evidence that the model predicts every client's future movement well.

I inspect coefficients and three high-confidence mistakes below. The examples deliberately omit IDs.

In [4]:
coefs = model.named_steps["logisticregression"].coef_[0]
importance = pd.DataFrame({"feature": features, "coefficient": coefs})
importance["absolute_size"] = importance["coefficient"].abs()
importance = importance.sort_values("absolute_size", ascending=False)
print("Feature directions after standardization:")
print(importance[["feature", "coefficient"]].round(3).to_string(index=False))

test["prediction"] = (test["model_score"] >= 0.5).astype(int)
test["error_type"] = np.select(
    [(test["prediction"] == 1) & (test["is_future_decline"] == 0),
     (test["prediction"] == 0) & (test["is_future_decline"] == 1)],
    ["false positive", "false negative"], default="correct",
)
wrong = test[test["error_type"] != "correct"].copy()
wrong["confidence"] = np.where(wrong["prediction"] == 1, wrong["model_score"], 1 - wrong["model_score"])
examples = wrong.nlargest(3, "confidence")[[
    "error_type", "model_score", "log_early_impressions", "early_ctr",
    "early_active_days", "early_avg_position", "early_position_volatility",
]].round(3)
print("\nThree difficult mistakes (IDs omitted):")
print(examples.to_string(index=False))

error_summary = wrong.groupby("error_type").agg(
    n=("is_future_decline", "size"), median_position=("early_avg_position", "median"),
    median_impressions=("early_impressions", "median"),
).reset_index().round(2)
print("\nError summary:")
print(error_summary.to_string(index=False))

output_dir = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
metrics = {
    "split_seed": 42,
    "train_rows": int(len(train)), "test_rows": int(len(test)),
    "train_clients": int(train["client_hash_id"].nunique()),
    "test_clients": int(test["client_hash_id"].nunique()),
    "test_base_rate": round(float(test["is_future_decline"].mean()), 6),
    "baseline_precision_at_100": round(float(precision_at_k(test, "baseline_score", 100)), 6),
    "model_precision_at_100": round(float(precision_at_k(test, "model_score", 100)), 6),
    "model_roc_auc": round(float(roc_auc_score(test["is_future_decline"], test["model_score"])), 6),
}
with open(output_dir / "ml08_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nSaved aggregate metrics receipt.")

Feature directions after standardization:
                  feature  coefficient
                early_ctr       -0.364
       early_avg_position        0.093
        early_active_days        0.029
    log_early_impressions        0.028
early_position_volatility       -0.018

Three difficult mistakes (IDs omitted):
    error_type  model_score  log_early_impressions  early_ctr  early_active_days  early_avg_position  early_position_volatility
false negative        0.002                  4.673      0.075                 15               4.019                      3.306
false negative        0.009                  6.836      0.056                 15               2.757                      0.464
false negative        0.014                  5.908      0.049                 13               4.223                      1.805

Error summary:
    error_type    n  median_position  median_impressions
false negative 3659             7.42               442.0
false positive   46            67.40     

## Self-check

- [x] The method fits a binary ranking problem and stays deliberately simple
- [x] Entire clients are held out and the seed is fixed
- [x] Baseline and model use the same rows, split, labels, and precision@K metrics
- [x] The comparison table includes the base rate
- [x] Feature directions and real error examples are inspected
- [x] IDs are omitted from displayed examples
- [x] The result is described as observed decision support, not causal proof
- [x] The notebook runs top to bottom without errors